# Atelier : comprendre le function calling / tool calling

Objectif du notebook : comprendre concrètement comment un LLM appelle une fonction externe. Dans app_TD2_tool_calling.py on fait cela dans le 'format argent', ici le but n'est pas de 
rentrer dans la mecanique agent mais de comprendre comment bien et facilement faire des fonctions.

 https://developers.openai.com/api/docs/guides/function-calling
 TODO: rajouter d'autres liens sur le function-calling

On va progresser en 5 étapes :

1. Écrire une fonction Python normale.
2. Décrire cette fonction avec un JSON Schema.
3. La rendre appelable par OpenAI en mode function calling.
4. Construire une mini boucle agentique.
5. Reproduire une logique similaire avec Ollama.

> Idée clé : le modèle n'exécute pas directement la fonction. Il produit une demande d'appel structurée. C'est notre code Python qui exécute réellement la fonction.

## Prérequis

Installez les librairies nécessaires si besoin :

In [ ]:
# À lancer une seule fois si nécessaire
# %pip install openai ollama

## Configuration OpenAI

Pour utiliser OpenAI, on peut mettre notre clé API dans une variable d'environnement :

```bash
export OPENAI_API_KEY="votre-cle-api"
```

ou bien on lit directement depuis le openai_api_key.txt (mais pas très safe + pensé à l'inclure dans le .gitignore)

In [4]:
import os
import json
from pprint import pprint


if ("OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"].strip()):
    with open("openai_api_key.txt", "r") as f:
        api_key = f.read().strip()
    os.environ["OPENAI_API_KEY"] = api_key
else:
    api_key = os.environ["OPENAI_API_KEY"]

# 1. Exemple basique de fonction

On commence avec une fonction très simple. Noter que c'est une bonne pratique ici de typer nos fonctions, car notamment comme le function calling passe par l'échange de JSON schema, on ne peut pas placer n'importe quel type d'objet en argument.


In [4]:
def add(a: int, b: int) -> dict:
    """Additionne deux entiers."""
    return {"result": a + b}

add(37, 58)

{'result': 95}


Typiquement on voudrait donc que lors du prompt suivant le modèle soit capable d'appeler la fonction `add`

> Combien font 37 + 58 ?

et on voudrait que le LLM renvoie le json suivant 

```json
{
  "name": "add",
  "arguments": {
    "a": 37,
    "b": 58
  }
}
```

cela nécessite donc que le LLM connaissance la fonction (que sans native function calling on met juste dans le prompt) et soit assez bon pour bien formater le json.

# 2. Description de la fonction avec un JSON Schema

On peut donc mettre directement les outils dans le prompt, mais il s'agit plus d'un hack que d'une solution durable. En effet, imaginons que l'on ait une fonction complexe et potentiellement très longue : on polluerait notre fenêtre de contexte avec de nombreuses lignes de code. Et finalement, ce n'est pas tellement un problème de taille de contexte avec les LLMs modernes, qu'une utilisation inefficace de l'information. On se moque des détails internes de la fonction : idéalement, la docstring de la fonction — et en pratique souvent moins que ça — suffit au LLM pour comprendre quand utiliser cette fonction.

Le point important est que le modèle n’a pas besoin de connaître l’implémentation complète de l’outil : il a surtout besoin de connaître son rôle, ses paramètres d’entrée et le format attendu de son résultat. Donc typiquement quand on veut que le modèle ne voit pas notre code Python. On veut lui passer une description structurée de l'outil.

Cette description contient généralement :

- le nom de la fonction ;
- une description courte : qui est tres importante.
- les paramètres attendus ;
- les types des paramètres ;
- les paramètres obligatoires.

In [5]:
add_tool_schema = {
    "type": "function",
    "function": {
        "name": "add",
        "description": "Add two integers and return the result.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "integer",
                    "description": "The first integer."
                },
                "b": {
                    "type": "integer",
                    "description": "The second integer."
                }
            },
            "required": ["a", "b"],
            "additionalProperties": False
        },
        "strict": True
    }
}

pprint(add_tool_schema)

{'function': {'description': 'Add two integers and return the result.',
              'name': 'add',
              'parameters': {'additionalProperties': False,
                             'properties': {'a': {'description': 'The first '
                                                                 'integer.',
                                                  'type': 'integer'},
                                            'b': {'description': 'The second '
                                                                 'integer.',
                                                  'type': 'integer'}},
                             'required': ['a', 'b'],
                             'type': 'object'},
              'strict': True},
 'type': 'function'}


Et maintenant typiquement, on fait comme cela pour passer l'outils au LLM (dans le cas de OpenAI) lorsque l'on va faire le prompt 
> Combien font 37 + 58 ?

In [8]:
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 97.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [openai]15/16 [openai]


In [ ]:
from openai import OpenAI

client = OpenAI()

messages = [
    {"role": "user", "content": "Combien font 37 + 58 ?"}
]

tools = [add_tool_schema]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)

assistant_message = response.choices[0].message
pprint(assistant_message.model_dump())

{'annotations': [],
 'audio': None,
 'content': None,
 'function_call': None,
 'refusal': None,
 'role': 'assistant',
 'tool_calls': [{'function': {'arguments': '{"a":37,"b":58}', 'name': 'add'},
                 'id': 'call_6GFptmkNygYCrzfiCAVzi5K1',
                 'type': 'function'}]}


Et maintenant avant d'analyser l'output du prompt (on voit deja qu'il a bien reussi a faire l'appel a la fonction `add` avec les bons arguments), faisons plusieurs remarques
sur le format de `add_tool_schema` (qui est un dictionnaire python et pas tout a fait un json) :
- La structure externe est une convention OpenAI.
- La partie "parameters" est du JSON Schema.

C'est a dire que la partie suivante est une format attendu par OpenAI
```json
{
    "type": "function",
    "function": {
        "name": "...",
        "description": "...",
        "parameters": { ... },
        "strict": True
    }
}
```
A noter d'ailleurs que pour une fonction le type est toujours "function" (voir https://developers.openai.com/api/docs/guides/function-calling pour plus de details).
En effet OpenAI distingue les function tools d’autres formes de tools ou de sorties structurées.
Par contre la partie suivante est un JSON SCHEMA qui est **la description de la forme que doit avoir un JSON.**
```json
"parameters": {
    "type": "object",
    "properties": {
        "a": {
            "type": "integer",
            "description": "The first integer."
        },
        "b": {
            "type": "integer",
            "description": "The second integer."
        }
    },
    "required": ["a", "b"],
    "additionalProperties": False
}
```
Et donc ici par exemple le JSON suivant est valide
```
{
  "a": 3,
  "b": 5
}
```
mais pas celui la
```
{
  "a": 3,
}
```
JSON Schema (https://json-schema.org/specification) est donc une sorte de contrat de validation pour des données JSON. Le site officiel le présente comme un vocabulaire permettant la cohérence, la validité et l’interopérabilité des données JSON. Noter aussi que `"additionalProperties": False` signifie que aucun autre champ n’est accepté dans la fonction.

JSON Schema est une spécification ouverte, maintenue par une communauté open source autour du projet JSON Schema.  
OpenAI ne l’a pas inventée : OpenAI l’utilise parce que c’est déjà une manière standard de décrire la forme attendue d’un objet JSON.

# 3. Utiliser OpenAI avec function calling

Dans cette section, on envoie au modèle :

1. le message utilisateur ;
2. la liste des tools disponibles.

Le modèle peut alors soit répondre directement, soit demander un appel d'outil.

In [10]:
from openai import OpenAI

client = OpenAI()

messages = [
    {"role": "user", "content": "Combien font 37 + 58 ?"}
]

tools = [add_tool_schema]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)

assistant_message = response.choices[0].message
pprint(assistant_message.model_dump())

{'annotations': [],
 'audio': None,
 'content': None,
 'function_call': None,
 'refusal': None,
 'role': 'assistant',
 'tool_calls': [{'function': {'arguments': '{"a":37,"b":58}', 'name': 'add'},
                 'id': 'call_eXVmhIR4pm8lfmZB5OP5BtNp',
                 'type': 'function'}]}


Si tout se passe bien, le message assistant contient un champ `tool_calls`.

Ce champ est la demande du modèle :

> Je veux appeler telle fonction avec tels arguments.

Il ne s'agit pas encore du résultat de la fonction. Ensuite il faut parser la réponse et organiser l'appel à la fonction. On vérifie que le parsing est correct.

In [11]:
if assistant_message.tool_calls:
    tool_call = assistant_message.tool_calls[0]
    print("Nom du tool demandé :", tool_call.function.name)
    print("Arguments bruts :", tool_call.function.arguments)
    print("Arguments parsés :")
    pprint(json.loads(tool_call.function.arguments))
else:
    print("Le modèle n'a pas demandé de tool call.")
    print("Réponse directe :", assistant_message.content)

Nom du tool demandé : add
Arguments bruts : {"a":37,"b":58}
Arguments parsés :
{'a': 37, 'b': 58}


Maintenant pour avoir une boucle complète, il faut faire les actions suivantes :

1. récupérer le nom de la fonction ;
2. parser les arguments ;
3. trouver la vraie fonction Python correspondante ;
4. l'exécuter ;
5. renvoyer le résultat au modèle.

Pour l'exécuter il suffit juste de bien savoir lire le résultat `assistant_message`

In [13]:
available_functions = {
    "add": add
}

messages.append(assistant_message)

if assistant_message.tool_calls:
    for tool_call in assistant_message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        print("Exécution de", function_name, "avec", arguments)

        function_to_call = available_functions[function_name]
        result = function_to_call(**arguments)

        print("Résultat du tool :", result)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

Exécution de add avec {'a': 37, 'b': 58}
Résultat du tool : {'result': 95}


Et en ajoutant le résultat des appels dans la liste message, on peut demander au modele de nous renvoyer la réponse finale

In [14]:
final_response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)

print(final_response.choices[0].message.content)

37 + 58 font 95.


Bien sûr, on ne fait ici qu’effleurer la surface. Mais finalement, l’agentique commence souvent par la complexification de ce schéma de base : un modèle qui peut décider d’appeler une fonction, un programme qui exécute cette fonction, puis un modèle qui reprend la main à partir du résultat.

À partir de là, on peut progressivement ajouter :

- l’accès à davantage d’outils, de natures différentes : API, base de données, moteur de recherche, fichiers, navigateur, calendrier, email, etc. ;
- une gestion plus avancée de la boucle : le modèle peut appeler un outil, observer le résultat, décider d’en appeler un autre, puis continuer jusqu’à produire une réponse finale ;
- de la mémoire ou un historique de travail ;
- des validations et des garde-fous avant d’exécuter certaines actions ;
- des retries lorsque les arguments sont incorrects ou que l’outil échoue ;
- éventuellement plusieurs modèles spécialisés : un modèle pour planifier, un autre pour appeler les outils, un autre pour vérifier le résultat.

Autrement dit, le function calling est une première brique vers l’agentique : il donne au modèle un moyen structuré d’agir sur le monde extérieur, tout en laissant au programme la responsabilité d’exécuter réellement les actions.

# 4. **Exercice** : Ecrire le JSON SCHEMA pour une fonction donnee

On ajoute maintenant une fonction qui prend un argument extrait depuis le langage naturel. 


In [ ]:
def get_weather(city: str, unit: str = "celsius") -> dict:
    """Retourne une météo fictive pour quelques villes."""
    fake_weather = {
        "Paris": {"temperature": 18, "condition": "nuageux"},
        "Lyon": {"temperature": 21, "condition": "ensoleillé"},
        "Marseille": {"temperature": 24, "condition": "venteux"},
        "Tokyo": {"temperature": 20, "condition": "pluvieux"},
    }

    data = fake_weather.get(city)
    if data is None:
        return {"error": "city_not_found", "city": city}

    return {
        "city": city,
        "unit": unit,
        **data
    }

get_weather("Marseille")

**Exercice** : Ecrire le `weather_tool_schema` pour decrire la fonction `get_weather`.

**Exercice** : Utiliser `gpt-4.1-mini` pour repondre au prompt suivant
```text
Quel temps fait-il à Marseille ?
```

Exécution du tool demandé :

# 5. Exercice : écrire un outil pour répondre à une question difficile sans calcul fiable

On souhaite répondre au prompt suivant :

> J’ai acheté 3 articles :
> - un clavier à 79,90 €
> - une souris à 39,50 €
> - un écran à 249,99 €
>
> J’ai un code promo de 12 %, puis je dois ajouter une TVA de 20 %.
> Les frais de livraison sont de 8,90 €, mais ils sont offerts si le total TTC avant livraison dépasse 350 €.
>
> Quel est le montant final à payer ?

Commencez par essayer de répondre sans outil.  
Puis écrivez un outil Python qui calcule le total de manière fiable.

Cet exercice illustre une idée importante : un outil n’est pas forcément une API externe.  
Un outil peut simplement être une fonction Python locale qui applique une règle métier de manière fiable.  
Le rôle du LLM est alors d’identifier les bons paramètres à partir du langage naturel, puis de demander l’appel de la fonction.

# 6. Version Ollama

Ollama propose aussi un champ `tools` dans son API de chat pour les modèles compatibles tool calling.

Avant de lancer cette section :

1. Installer Ollama.
2. Lancer un modèle compatible, par exemple :

```bash
ollama pull qwen3
```

ou un autre modèle récent supportant les tools.

> Attention : tous les modèles Ollama ne sont pas aussi bons en tool calling. La qualité dépend beaucoup du modèle et de son template.

In [ ]:
# Test d'import Ollama
try:
    import ollama
    print("Librairie ollama disponible.")
except ImportError:
    print("Librairie ollama non installée. Lancez : %pip install ollama")

In [ ]:
# Exemple minimal avec Ollama
# Décommentez pour tester si Ollama tourne localement.

# import ollama
#
# ollama_tools = [
#     {
#         "type": "function",
#         "function": {
#             "name": "get_weather",
#             "description": "Get the fake weather for a city.",
#             "parameters": {
#                 "type": "object",
#                 "properties": {
#                     "city": {
#                         "type": "string",
#                         "description": "The city name."
#                     },
#                     "unit": {
#                         "type": "string",
#                         "enum": ["celsius", "fahrenheit"],
#                         "description": "Temperature unit."
#                     }
#                 },
#                 "required": ["city", "unit"]
#             }
#         }
#     }
# ]
#
# response = ollama.chat(
#     model="qwen3",
#     messages=[
#         {"role": "user", "content": "Quel temps fait-il à Paris ?"}
#     ],
#     tools=ollama_tools
# )
#
# pprint(response)

## Exécution manuelle du tool avec Ollama

Selon la version de la librairie Ollama, la structure exacte de la réponse peut varier légèrement. L'idée reste identique :

1. récupérer `tool_calls` ;
2. lire le nom du tool ;
3. parser les arguments ;
4. exécuter la fonction Python ;
5. renvoyer un message avec `role: "tool"`.

In [ ]:
# Exemple de boucle Ollama à adapter selon la structure de réponse obtenue.
# Décommentez pour tester.

# import ollama
#
# def run_ollama_agent(user_input: str, model: str = "qwen3", max_steps: int = 5):
#     messages = [{"role": "user", "content": user_input}]
#
#     for step in range(max_steps):
#         print(f"\n--- Étape Ollama {step + 1} ---")
#
#         response = ollama.chat(
#             model=model,
#             messages=messages,
#             tools=ollama_tools
#         )
#
#         message = response["message"]
#         messages.append(message)
#
#         tool_calls = message.get("tool_calls", [])
#
#         if not tool_calls:
#             print("Réponse finale :")
#             print(message.get("content", ""))
#             return message.get("content", "")
#
#         for tool_call in tool_calls:
#             function = tool_call["function"]
#             function_name = function["name"]
#             arguments = function.get("arguments", {})
#
#             print("Tool demandé :", function_name)
#             print("Arguments :", arguments)
#
#             if function_name == "get_weather":
#                 result = get_weather(**arguments)
#             else:
#                 result = {"error": f"Unknown function: {function_name}"}
#
#             print("Résultat :", result)
#
#             messages.append({
#                 "role": "tool",
#                 "content": json.dumps(result, ensure_ascii=False),
#             })
#
#     print("Nombre maximum d'étapes atteint.")
#     return None
#
# run_ollama_agent("Quel temps fait-il à Paris ?")

# 6. Résumé à retenir

Le function calling, c'est :

```text
LLM → produit une intention structurée
Python → exécute la fonction réelle
LLM → utilise le résultat pour répondre
```

La fonction doit surtout respecter un contrat clair :

- nom explicite ;
- description courte ;
- paramètres simples ;
- JSON Schema propre ;
- retour sérialisable ;
- gestion d'erreurs ;
- garde-fous pour les actions sensibles.

Le native tool calling rend cette interface plus fiable qu'un simple prompt demandant au modèle d'écrire du JSON.